In [1]:
import os
import urllib.request

base = "https://storage.data.gov.my/pricecatcher/"
files = ["lookup_item.parquet", "lookup_premise.parquet"] + [
    f"pricecatcher_2026-{m:02d}.parquet" for m in range(1, 7)
]

raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

for f in files:
    path = os.path.join(raw_dir, f)
    if os.path.exists(path):
        print("already have", f)
    else:
        urllib.request.urlretrieve(base + f, path)
        print("downloaded", f)

already have lookup_item.parquet
already have lookup_premise.parquet
already have pricecatcher_2026-01.parquet
already have pricecatcher_2026-02.parquet
already have pricecatcher_2026-03.parquet
already have pricecatcher_2026-04.parquet
already have pricecatcher_2026-05.parquet
already have pricecatcher_2026-06.parquet


In [2]:
import pandas as pd

item = pd.read_parquet("../data/raw/lookup_item.parquet")
premise = pd.read_parquet("../data/raw/lookup_premise.parquet")

print(item.shape, premise.shape)
display(item.head())
display(premise.head())

(797, 5) (3929, 6)


,item_code,item,unit,item_group,item_category
0,-1,NaN,NaN,NaN,NaN
1,1,AYAM BERSIH - STANDARD,1kg,BARANGAN SEGAR,AYAM
2,2,AYAM BERSIH - SUPER,1kg,BARANGAN SEGAR,AYAM
3,3,AYAM HIDUP,1kg,BARANGAN SEGAR,AYAM
4,9,DAGING KAMBING BEBIRI IMPORT BERTULANG (MUTTON...,1kg,BARANGAN SEGAR,DAGING


,premise_code,premise,address,premise_type,state,district
0,-1,Unknown Premise,Unknown Address,NaN,NaN,NaN
1,2,PASAR BESAR IPOH,"JALAN LAKSAMANA,TAMAN JUBILEE,30300 IPOH, PERAK",Pasar Basah,Perak,Kinta
2,3,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
3,6,KEDAI RUNCIT TAFAZ MAJU PRESINT 9,"NO.5, JALAN P9B/1,PRESINT 9,62250 PUTRAJAYA",Kedai Runcit,W.P. Putrajaya,Wp Putrajaya
4,7,KEDAI RUNCIT SYAZ MAJU PRESINT 9,"NO.5, JALAN P9E/1,PRESINT 9,62250 PUTRAJAYA",Kedai Runcit,W.P. Putrajaya,Wp Putrajaya


In [3]:
prices = pd.read_parquet("../data/raw/pricecatcher_2026-01.parquet")

print(prices.shape)
print(prices.dtypes)
print(prices.isna().sum())
print(prices["date"].min(), prices["date"].max())
print(prices["price"].describe())
print("duplicate rows:", prices.duplicated(["date", "premise_code", "item_code"]).sum())

(1976483, 4)
date             object
premise_code      int64
item_code         int64
price           float64
dtype: object
date            0
premise_code    0
item_code       0
price           0
dtype: int64
2026-01-01 2026-01-31
count    1.976483e+06
mean     1.252541e+01
std      1.091877e+01
min      3.000000e-02
25%      5.400000e+00
50%      9.600000e+00
75%      1.528000e+01
max      1.418000e+03
Name: price, dtype: float64
duplicate rows: 0


In [4]:
import os
print(sorted(os.listdir("../data/raw")))

print(item["item_group"].value_counts(dropna=False))
print(item["item_category"].value_counts(dropna=False).head(30))
print(premise["premise_type"].value_counts(dropna=False))
print(sorted(premise["state"].dropna().unique()))

['lookup_item.parquet', 'lookup_premise.parquet', 'pricecatcher_2026-01.parquet', 'pricecatcher_2026-02.parquet', 'pricecatcher_2026-03.parquet', 'pricecatcher_2026-04.parquet', 'pricecatcher_2026-05.parquet', 'pricecatcher_2026-06.parquet']
item_group
BARANGAN BERBUNGKUS         219
MAKANAN SIAP MASAK          191
BARANGAN SEGAR              164
MINUMAN                      50
SUSU DAN BARANGAN BAYI       50
BARANGAN KEDAI SERBANEKA     49
PRODUK KEBERSIHAN            46
BARANGAN KERING              27
NaN                           1
Name: count, dtype: int64
item_category
LAUK                               72
MINUMAN                            47
SAYUR-SAYURAN                      41
BAHAN LAUT                         40
MINYAK DAN LEMAK                   39
DAGING                             32
REMPAH RATUS (BERBUNGKUS)          30
MEE / BIHUN / KUEY TEOW            28
SUSU BAYI                          27
LAIN-LAIN                          27
TERSEDIA MINUM                     26
B

In [5]:
print("items not in lookup:", (~prices["item_code"].isin(item["item_code"])).sum())
print("premises not in lookup:", (~prices["premise_code"].isin(premise["premise_code"])).sum())
print("item_code -1 rows:", (prices["item_code"] == -1).sum())
print("premise_code -1 rows:", (prices["premise_code"] == -1).sum())

print(prices.groupby("date").size())

med = prices.groupby("item_code")["price"].transform("median")
ratio = prices["price"] / med
print("price > 5x item median:", (ratio > 5).sum())
print("price < 0.2x item median:", (ratio < 0.2).sum())
print(prices.assign(ratio=ratio).sort_values("ratio", ascending=False).head(10))

items not in lookup: 31242
premises not in lookup: 0
item_code -1 rows: 0
premise_code -1 rows: 0
date
2026-01-01     10757
2026-01-02     38294
2026-01-03     38318
2026-01-04     39038
2026-01-05    211381
2026-01-06     37851
2026-01-07     38264
2026-01-08     38385
2026-01-09     38688
2026-01-10     38471
2026-01-11     38719
2026-01-12    189192
2026-01-13     37700
2026-01-14     36909
2026-01-15     38728
2026-01-16     39149
2026-01-17     32971
2026-01-18     37730
2026-01-19    192538
2026-01-20     38790
2026-01-21     39092
2026-01-22     39079
2026-01-23     39196
2026-01-24     39217
2026-01-25     39542
2026-01-26    214559
2026-01-27     38466
2026-01-28     39219
2026-01-29    197554
2026-01-30     39469
2026-01-31     39217
dtype: int64
price > 5x item median: 40
price < 0.2x item median: 298
               date  premise_code  item_code   price       ratio
1325785  2026-01-23         21557        109   600.0  150.000000
1325801  2026-01-23         21557       2086  

In [6]:
missing = prices[~prices["item_code"].isin(item["item_code"])]
print("rows:", len(missing), "| distinct item codes:", missing["item_code"].nunique())
print("share of January rows:", round(len(missing) / len(prices) * 100, 2), "%")
print("max code in lookup:", item["item_code"].max())
print("missing codes range:", missing["item_code"].min(), "to", missing["item_code"].max())
print(missing["item_code"].value_counts().head(15))
print(missing["price"].describe())

rows: 31242 | distinct item codes: 14
share of January rows: 1.58 %
max code in lookup: 2115
missing codes range: 2023 to 2096
item_code
2057    3609
2094    3421
2059    3328
2053    3209
2095    2952
2089    2930
2090    2496
2091    2112
2023    1915
2092    1908
2096    1758
2093    1566
2035      20
2048      18
Name: count, dtype: int64
count    31242.000000
mean         8.519072
std          3.848469
min          1.600000
25%          5.900000
50%          7.200000
75%         10.500000
max         56.900000
Name: price, dtype: float64


In [7]:
print([repr(x) for x in premise["premise_type"].unique()])

p = prices.merge(premise[["premise_code", "premise_type"]], on="premise_code", how="left")
p["date"] = pd.to_datetime(p["date"])
daily = p.groupby(["date", "premise_type"]).size().unstack(fill_value=0)
spike_days = pd.to_datetime(["2026-01-04", "2026-01-05", "2026-01-06", "2026-01-29"])
display(daily.loc[spike_days])

['nan', "'Pasar Basah '", "'Pasar Raya / Supermarket'", "'Kedai Runcit'", "'Pasar Mini'", "'Restoran India Muslim'", "'Restoran Cina'", "'Hypermarket'", "'Restoran Melayu'", "'Medan Selera'", "'Foodcourt'", "'Pasar Basah'", "'Borong'", "'Kedai Serbaneka'"]


premise_type,Borong,Hypermarket,Kedai Runcit,Pasar Basah,Pasar Basah,Pasar Mini,Pasar Raya / Supermarket
2026-01-04,0,5055,88,403,10653,606,22233
2026-01-05,0,25933,19119,589,14366,45919,105455
2026-01-06,0,4882,89,395,10070,578,21837
2026-01-29,0,23923,16768,665,17559,40954,97685


In [8]:
med = prices.groupby("item_code")["price"].transform("median")
out = prices[prices["price"] / med > 5].copy()
out["median"] = med[out.index]
out["price_div_100_vs_median"] = (out["price"] / 100) / out["median"]
print(out["price_div_100_vs_median"].describe())
print(out["premise_code"].value_counts().head(10))

count    40.000000
mean      0.433763
std       0.508490
min       0.051026
25%       0.064950
50%       0.099896
75%       0.975815
max       1.500000
Name: price_div_100_vs_median, dtype: float64
premise_code
16519    5
21640    4
21557    4
1291     3
3204     3
21231    2
14564    1
14663    1
21378    1
5282     1
Name: count, dtype: int64


In [9]:
from pathlib import Path
import pandas as pd

raw = Path("../data/raw")
item = pd.read_parquet(raw / "lookup_item.parquet")
premise = pd.read_parquet(raw / "lookup_premise.parquet")
premise["premise_type"] = premise["premise_type"].str.strip()
print("premise_code unique:", premise["premise_code"].is_unique)

item_codes = set(item["item_code"])
premise_codes = set(premise["premise_code"])
ptype = premise.set_index("premise_code")["premise_type"]

rows, type_counts = [], {}
for m in range(1, 7):
    label = f"2026-{m:02d}"
    df = pd.read_parquet(raw / f"pricecatcher_{label}.parquet")
    ratio = df["price"] / df.groupby("item_code")["price"].transform("median")
    rows.append({
        "month": label,
        "rows": len(df),
        "days": df["date"].nunique(),
        "premises": df["premise_code"].nunique(),
        "items": df["item_code"].nunique(),
        "nulls": int(df.isna().sum().sum()),
        "dup_keys": int(df.duplicated(["date", "premise_code", "item_code"]).sum()),
        "price<=0": int((df["price"] <= 0).sum()),
        "item_not_in_lookup": int((~df["item_code"].isin(item_codes)).sum()),
        "premise_not_in_lookup": int((~df["premise_code"].isin(premise_codes)).sum()),
        "item_-1": int((df["item_code"] == -1).sum()),
        "premise_-1": int((df["premise_code"] == -1).sum()),
        "high_>5x": int((ratio > 5).sum()),
        "low_<0.2x": int((ratio < 0.2).sum()),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
    })
    type_counts[label] = df["premise_code"].map(ptype).fillna("unmatched").value_counts()
    del df

display(pd.DataFrame(rows).set_index("month").T)
display(pd.DataFrame(type_counts).fillna(0).astype(int))

premise_code unique: True


month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
rows,1976483,1369552,1547695,1785221,1717713,1763560
days,31,25,25,30,29,27
premises,2139,2126,2126,2124,2130,2127
items,343,341,343,340,343,353
nulls,0,0,0,0,0,0
dup_keys,0,0,0,0,0,0
price<=0,0,0,0,0,0,0
item_not_in_lookup,31242,18669,24205,24001,26315,30009
premise_not_in_lookup,0,0,0,0,0,0
item_-1,0,0,0,0,0,0


,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
premise_code,,,,,,
Borong,6,201,0,0,0,0
Hypermarket,250092,171031,193267,226309,220145,224411
Kedai Runcit,90949,54033,73395,68745,73881,82165
Pasar Basah,348477,263140,274972,342725,310868,305413
Pasar Mini,228225,137599,180051,182619,187017,216454
Pasar Raya / Supermarket,1058734,743548,826010,964823,925802,935117


In [10]:
df = pd.read_parquet(raw / "pricecatcher_2026-01.parquet")
df["ratio"] = df["price"] / df.groupby("item_code")["price"].transform("median")
low = df[df["ratio"] < 0.2].merge(item[["item_code", "item", "unit"]], on="item_code", how="left")

print(len(low))
print(low["item"].value_counts(dropna=False).head(10))
print(low["premise_code"].value_counts().head(10))
display(low.sort_values("ratio").head(10))

298
item
SAWI HIJAU                                          94
SAWI PENDEK/JEPUN/SIOW PAK CHOY (1KG)               45
KAILAN                                              20
KACANG BENDI                                        17
BAYAM HIJAU                                         12
IKAN SELAR KUNING (≥ 11 EKOR SEKILOGRAM)             9
TIMUN                                                7
IKAN KERISI (ANTARA 5 HINGGA 10 EKOR SEKILOGRAM)     5
KANGKUNG                                             5
SOTONG (ANTARA 11 HINGGA17 EKOR SEKILOGRAM)          4
Name: count, dtype: int64
premise_code
21410    22
18344    15
15192     9
17771     8
5488      7
21690     7
21097     7
656       7
3984      6
21467     6
Name: count, dtype: int64


,date,premise_code,item_code,price,ratio,item,unit
267,2026-01-30,234,70,0.08,0.004444,IKAN SELAR/PELATA (≤ 7 EKOR SEKILOGRAM),1kg
10,2026-01-04,6931,129,0.03,0.007792,BAWANG BESAR KUNING/HOLLAND,1kg
188,2026-01-26,10720,2042,0.28,0.009689,PETPET PANTS GOLD+,"M56, L46, XL38"
213,2026-01-27,11699,69,0.13,0.010008,IKAN SELAR KUNING (≥ 11 EKOR SEKILOGRAM),1kg
149,2026-01-23,16317,96,0.10,0.010010,KACANG BENDI,1kg
71,2026-01-11,15786,1481,0.13,0.014607,KUBIS BUNGA (CAULIFLOWER),1kg
104,2026-01-16,18116,1111,0.60,0.046912,TELUR AYAM GRED C,30 biji
166,2026-01-24,20814,109,0.20,0.050000,LOBAK MERAH,1kg
225,2026-01-28,12761,1819,0.50,0.050050,LENGKUAS,1kg
15,2026-01-05,11632,229,0.10,0.058824,F&N OREN (TIN),325 ml


In [11]:
present = set()
for m in range(1, 7):
    d = pd.read_parquet(raw / f"pricecatcher_2026-{m:02d}.parquet", columns=["date"])["date"]
    present |= set(pd.to_datetime(d.unique()))

full = pd.date_range("2026-01-01", "2026-06-30")
missing_days = [d for d in full if d not in present]
print(len(missing_days), "missing days")
for d in missing_days:
    print(d.date(), d.day_name())

14 missing days
2026-02-13 Friday
2026-02-17 Tuesday
2026-02-18 Wednesday
2026-03-14 Saturday
2026-03-19 Thursday
2026-03-20 Friday
2026-03-21 Saturday
2026-03-22 Sunday
2026-03-23 Monday
2026-05-01 Friday
2026-05-27 Wednesday
2026-06-01 Monday
2026-06-17 Wednesday
2026-06-20 Saturday


In [14]:
frames = []
for m in range(1, 7):
    df = pd.read_parquet(raw / f"pricecatcher_2026-{m:02d}.parquet", columns=["item_code"])
    frames.append(df["item_code"].value_counts().rename(f"2026-{m:02d}"))

counts = pd.concat(frames, axis=1).fillna(0).astype(int)
counts["total_rows"] = counts.sum(axis=1)
counts = counts.join(item.set_index("item_code")[["item", "item_group", "item_category"]], how="left")

print(counts["item_group"].value_counts(dropna=False))
print(counts.groupby("item_group", dropna=False)["total_rows"].sum().sort_values(ascending=False))
print(counts["item_category"].value_counts(dropna=False).head(25))

item_group
BARANGAN SEGAR            122
BARANGAN BERBUNGKUS       109
MINUMAN                    31
PRODUK KEBERSIHAN          27
SUSU DAN BARANGAN BAYI     26
BARANGAN KERING            24
NaN                        14
Name: count, dtype: int64
item_group
BARANGAN SEGAR            5931738
BARANGAN BERBUNGKUS       1875541
BARANGAN KERING           1139008
SUSU DAN BARANGAN BAYI     521318
MINUMAN                    271939
PRODUK KEBERSIHAN          266239
NaN                        154441
Name: total_rows, dtype: int64
item_category
SAYUR-SAYURAN                37
MINYAK DAN LEMAK             33
BAHAN LAUT                   27
DAGING                       22
TERSEDIA MINUM               20
BERAS                        19
BUAH-BUAHAN                  16
SUSU BAYI                    15
PENJAGAAN DIRI               15
REMPAH RATUS (BERBUNGKUS)    14
NaN                          14
PENJAGAAN RUMAH              12
BAHAN-BAHAN MINUMAN          11
BAWANG                       10
SAPUAN (SPR

In [15]:
frames = []
for m in range(1, 7):
    frames.append(pd.read_parquet(raw / f"pricecatcher_2026-{m:02d}.parquet",
                                  columns=["item_code", "premise_code"]).drop_duplicates())
ip = pd.concat(frames).drop_duplicates()
premises_per_item = ip.groupby("item_code")["premise_code"].nunique().rename("premises")

month_cols = [f"2026-{m:02d}" for m in range(1, 7)]
summary = counts.copy()
summary["months_present"] = (summary[month_cols] > 0).sum(axis=1)
summary = summary.join(premises_per_item).join(item.set_index("item_code")["unit"])

candidates = ["BERAS", "MINYAK DAN LEMAK", "TELUR", "AYAM", "BAWANG", "TEPUNG", "DAGING"]
cand = summary[summary["item_category"].isin(candidates)]
display(
    cand.sort_values(["item_category", "premises"], ascending=[True, False])
        .groupby("item_category").head(8)
        [["item_category", "item", "unit", "months_present", "premises", "total_rows"]]
)

,item_category,item,unit,months_present,premises,total_rows
item_code,,,,,,
1,AYAM,AYAM BERSIH - STANDARD,1kg,6,1261,138342
1804,AYAM,KEPAK AYAM (CHICKEN WING),1kg,6,1223,26197
1553,AYAM,WHOLE LEG AYAM,1kg,6,1162,24039
1550,AYAM,PAHA AYAM (CHICKEN DRUMSTICK),1kg,6,940,17827
1551,AYAM,DADA AYAM (CHICKEN KEEL),1kg,6,807,14707
1552,AYAM,THIGH AYAM,1kg,6,624,11407
2,AYAM,AYAM BERSIH - SUPER,1kg,6,431,40322
129,BAWANG,BAWANG BESAR KUNING/HOLLAND,1kg,6,2006,188818
1564,BAWANG,BAWANG PUTIH IMPORT (CHINA),1kg,6,1980,185553


In [16]:
candidates = ["BERAS", "MINYAK DAN LEMAK", "TELUR", "AYAM", "BAWANG", "TEPUNG", "DAGING"]

eligible = summary[
    summary["item_category"].isin(candidates)
    & (summary["months_present"] == 6)
    & (summary["premises"] >= 500)
]
basket = (
    eligible.sort_values(["item_category", "premises"], ascending=[True, False])
    .groupby("item_category").head(4)
    .reset_index()[["item_code", "item_category", "item", "unit", "premises", "total_rows"]]
)

print(len(basket), "items")
print(basket.groupby("item_category").size())
print(basket["unit"].isna().sum(), "items with missing unit")
display(basket)
basket.to_csv("../docs/basket_items.csv", index=False)

21 items
item_category
AYAM                4
BAWANG              4
BERAS               3
MINYAK DAN LEMAK    4
TELUR               3
TEPUNG              3
dtype: int64
0 items with missing unit


,item_code,item_category,item,unit,premises,total_rows
0,1,AYAM,AYAM BERSIH - STANDARD,1kg,1261,138342
1,1804,AYAM,KEPAK AYAM (CHICKEN WING),1kg,1223,26197
2,1553,AYAM,WHOLE LEG AYAM,1kg,1162,24039
3,1550,AYAM,PAHA AYAM (CHICKEN DRUMSTICK),1kg,940,17827
4,129,BAWANG,BAWANG BESAR KUNING/HOLLAND,1kg,2006,188818
5,1564,BAWANG,BAWANG PUTIH IMPORT (CHINA),1kg,1980,185553
6,1440,BAWANG,BAWANG BESAR IMPORT (INDIA),1kg,1770,149878
7,1442,BAWANG,BAWANG KECIL MERAH BIASA IMPORT (INDIA),1kg,1473,108845
8,2066,BERAS,BERAS PUTIH CAP FAIZA EMAS (IMPORT),10 kg,1067,19304
9,2068,BERAS,BERAS CAP JASMINE SUPER 5 SPECIAL (IMPORT),10 kg,879,15834
